# Imports

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES, load_segmenter
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import BOUNDARY_IOU_DILATION_PX, label_band
from uavloc.segmentation.eval import bilinear_argmax, lraspp_outputs, load_val_sets
from uavloc.track import REPO_ROOT, experiment_dir

# Params

In [ ]:
EXP = "016-boundary-loss"
MODELS = {
    "original": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "control (lambda 0)": "experiments/016-boundary-loss/artifacts/model-lambda0-seed42-steps400.pt",
    "boundary CE (lambda 3)": "experiments/016-boundary-loss/artifacts/model-lambda3-seed42-steps400.pt",
}
ZOOMS = [("bushcha", "2024-06-06", "building"), ("bushcha", "2024-06-06", "track"),
         ("birky-strip-google", "google", "building"), ("birky-strip-google", "google", "track")]
ZOOM_PX = 192
DEVICE = "cpu"
FIGURE_NAME = "zoom-crops.png"
FIGURE_DPI = 110

# Data

In [ ]:
val_sets = load_val_sets()
images = {(name, a.date): a for name, items in val_sets.items() for a in items}

In [ ]:
def densest_window(label: np.ndarray, cls: int, size_px: int) -> tuple[int, int]:
    """Top-left of the `size_px` square holding the most label-boundary pixels of class `cls`."""
    band = label_band(label, cls, BOUNDARY_IOU_DILATION_PX).astype(np.float32)
    density = cv2.boxFilter(band, -1, (size_px, size_px), normalize=False, anchor=(0, 0),
                            borderType=cv2.BORDER_CONSTANT)
    density = density[:label.shape[0] - size_px + 1, :label.shape[1] - size_px + 1]
    top, left = np.unravel_index(density.argmax(), density.shape)
    return int(top), int(left)

In [ ]:
windows = [densest_window(images[(s, d)].label, CLASSES.index(c), ZOOM_PX) for s, d, c in ZOOMS]
windows

# Model

In [ ]:
models = {name: load_segmenter(REPO_ROOT / path, DEVICE, architecture="lraspp") for name, path in MODELS.items()}

# Run

In [ ]:
needed = sorted({(s, d) for s, d, _ in ZOOMS})
preds = {(key, name): bilinear_argmax(lraspp_outputs(m, images[key].rgb, DEVICE), images[key].rgb)
         for key in needed for name, m in models.items()}

# Plots

In [ ]:
def colour(label: np.ndarray) -> np.ndarray:
    out = np.full((*label.shape, 3), 255, np.uint8)
    ok = label != IGNORE_LABEL
    out[ok] = SEG_PALETTE[label[ok]]
    return out

In [ ]:
fig, axes = plt.subplots(len(ZOOMS), 2 + len(MODELS), figsize=(3 * (2 + len(MODELS)), 3 * len(ZOOMS)), squeeze=False)
for row, ((s, d, c), (top, left)) in enumerate(zip(ZOOMS, windows)):
    window = np.s_[top:top + ZOOM_PX, left:left + ZOOM_PX]
    panels = [(images[(s, d)].rgb[window], f"{s} {d}\n{c} edges"), (colour(images[(s, d)].label[window]), "label")]
    panels += [(colour(preds[((s, d), name)][window]), name) for name in MODELS]
    for ax, (panel, title) in zip(axes[row], panels):
        ax.imshow(panel, interpolation="nearest"); ax.set_title(title, fontsize=9); ax.axis("off")
legend = [plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[k] / 255) for k in range(len(CLASSES))]
fig.legend(legend, CLASSES, loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .03, 1, 1))
fig.savefig(experiment_dir(EXP) / "figures" / FIGURE_NAME, dpi=FIGURE_DPI, bbox_inches="tight")